In [ ]:
# INIT DISHOP : créer la base initiale (à lancer une seule fois)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.firefox.options import Options
from selenium.webdriver import ActionChains
import pandas as pd
import time
import os
import glob

# Configuration
EMAIL = os.getenv("DISHOP_EMAIL", "your_email_here")
PASSWORD = os.getenv("DISHOP_PASSWORD", "your_password_here")

CREDENTIALS = [
    {
        "restaurant": "Chicken Street Pontault Combault",
        "email": EMAIL,
        "password": PASSWORD,
        "shop_id": "cs-pontault-combault"
    },
    {
        "restaurant": "Bangkok Factory Pontault Combault",
        "email": EMAIL,
        "password": PASSWORD,
        "shop_id": "bangkok-pontaultcombault"
    }
]

URL = "https://dashboard.dishop.co/"
EXPORT_BASENAME = "Liste_clients"
DOWNLOAD_DIR = os.getcwd()
FICHIER_CSV_INIT = "BDD_Client_Dishop.csv"
all_data = []


def create_driver():
    options = Options()
    options.headless = False
    options.set_preference("browser.download.folderList", 2)
    options.set_preference("browser.download.dir", DOWNLOAD_DIR)
    options.set_preference("browser.helperApps.neverAsk.saveToDisk", "text/csv")
    options.set_preference("pdfjs.disabled", True)
    return webdriver.Firefox(options=options)


def login(driver, wait, email, password, shop_id):
    driver.get(URL)
    wait.until(EC.presence_of_element_located((By.ID, "email"))).send_keys(email)
    driver.find_element(By.ID, "password").send_keys(password)
    driver.find_element(By.ID, "shopId").send_keys(shop_id)
    driver.find_element(By.XPATH, "//button[contains(text(), 'Connexion')]").click()


def clear_old_exports():
    for f in glob.glob(os.path.join(DOWNLOAD_DIR, f"{EXPORT_BASENAME}*.csv")):
        os.remove(f)


def wait_for_download(timeout=30):
    for _ in range(timeout):
        files = glob.glob(os.path.join(DOWNLOAD_DIR, f"{EXPORT_BASENAME}*.csv"))
        ready_files = [f for f in files if not f.endswith(".part")]
        if ready_files:
            return ready_files[0]
        time.sleep(1)
    return None


def collect_from_dishop(restaurant, email, password, shop_id):
    print(f"Traitement : {restaurant}")

    driver = create_driver()
    wait = WebDriverWait(driver, 30)

    try:
        login(driver, wait, email, password, shop_id)

        time.sleep(4)
        ActionChains(driver).move_by_offset(10, 10).click().perform()

        try:
            WebDriverWait(driver, 20).until(
                EC.invisibility_of_element_located((By.CSS_SELECTOR, ".modal.show"))
            )
        except:
            pass

        wait.until(EC.element_to_be_clickable((By.XPATH, "//a[@href='/clients']"))).click()
        time.sleep(2)

        clear_old_exports()

        export_btn = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Exporter la liste des clients')]"))
        )
        export_btn.click()

        downloaded_file = wait_for_download(timeout=30)

        if downloaded_file:
            df = pd.read_csv(downloaded_file)
            df["Restaurant"] = restaurant
            df["Plateforme"] = "Dishop"
            all_data.append(df)
            os.remove(downloaded_file)
            print(f"Données collectées : {restaurant}")
        else:
            print(f"Fichier non trouvé pour {restaurant}")

    except Exception as e:
        print(f"Erreur avec {restaurant} : {e}")
    finally:
        driver.quit()


# Traitement de tous les restaurants
for creds in CREDENTIALS:
    collect_from_dishop(creds["restaurant"], creds["email"], creds["password"], creds["shop_id"])

# Sauvegarde initiale
if all_data:
    df_init = pd.concat(all_data, ignore_index=True)
    df_init.to_csv(FICHIER_CSV_INIT, index=False, encoding="utf-8")
    print(f"Base initiale enregistrée dans : {FICHIER_CSV_INIT}")
else:
    print("Aucune donnée collectée.")

In [ ]:
# NOUVEAU SCRAPING DISHOP : récupérer les données récentes et les stocker temporairement

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.firefox.options import Options
from selenium.webdriver import ActionChains
import pandas as pd
import time
import os
import glob

# Configuration
EMAIL = os.getenv("DISHOP_EMAIL", "your_email_here")
PASSWORD = os.getenv("DISHOP_PASSWORD", "your_password_here")

CREDENTIALS = [
    {
        "restaurant": "Chicken Street Pontault Combault",
        "email": EMAIL,
        "password": PASSWORD,
        "shop_id": "cs-pontault-combault"
    },
    {
        "restaurant": "Bangkok Factory Pontault Combault",
        "email": EMAIL,
        "password": PASSWORD,
        "shop_id": "bangkok-pontaultcombault"
    }
]

URL = "https://dashboard.dishop.co/"
EXPORT_BASENAME = "Liste clients"
DOWNLOAD_DIR = os.getcwd()
FICHIER_CSV_TEMP = "Dishop_Temp.csv"
all_data = []


def create_driver():
    options = Options()
    options.set_preference("browser.download.folderList", 2)
    options.set_preference("browser.download.dir", DOWNLOAD_DIR)
    options.set_preference("browser.helperApps.neverAsk.saveToDisk", "text/csv")
    options.set_preference("pdfjs.disabled", True)
    options.set_preference("browser.download.useDownloadDir", True)
    return webdriver.Firefox(options=options)


def login(driver, wait, email, password, shop_id):
    driver.get(URL)
    wait.until(EC.presence_of_element_located((By.ID, "email"))).send_keys(email)
    driver.find_element(By.ID, "password").send_keys(password)
    driver.find_element(By.ID, "shopId").send_keys(shop_id)
    driver.find_element(By.XPATH, "//button[contains(text(), 'Connexion')]").click()


def clear_old_exports():
    for f in glob.glob(os.path.join(DOWNLOAD_DIR, f"{EXPORT_BASENAME}*.csv")):
        os.remove(f)


def wait_for_download(timeout=60):
    for i in range(timeout):
        files = glob.glob(os.path.join(DOWNLOAD_DIR, f"{EXPORT_BASENAME}*.csv"))
        ready_files = [f for f in files if not f.endswith(".part")]
        if ready_files:
            print(f"Fichier trouvé ({i + 1} sec) : {ready_files[0]}")
            return ready_files[0]
        time.sleep(1)
    return None


def collect_from_dishop(restaurant, email, password, shop_id):
    print(f"Traitement : {restaurant}")

    driver = create_driver()
    wait = WebDriverWait(driver, 30)

    try:
        login(driver, wait, email, password, shop_id)

        time.sleep(4)
        ActionChains(driver).move_by_offset(10, 10).click().perform()

        try:
            WebDriverWait(driver, 10).until(
                EC.invisibility_of_element_located((By.CLASS_NAME, "modal"))
            )
        except:
            pass

        wait.until(EC.element_to_be_clickable((By.XPATH, "//a[@href='/clients']"))).click()
        time.sleep(2)

        clear_old_exports()

        export_btn = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Exporter la liste des clients')]"))
        )
        export_btn.click()
        print("Export lancé, attente du fichier...")

        downloaded_file = wait_for_download(timeout=60)

        if downloaded_file:
            try:
                df = pd.read_csv(downloaded_file, sep=";", on_bad_lines="skip")
                df["Restaurant"] = restaurant
                df["Plateforme"] = "Dishop"
                all_data.append(df)
                print(f"Données collectées pour {restaurant} ({len(df)} lignes)")
            except Exception as e:
                print(f"Erreur de lecture du CSV ({restaurant}) : {e}")

            os.remove(downloaded_file)
        else:
            print(f"Fichier non trouvé pour {restaurant}")

    except Exception as e:
        print(f"Erreur avec {restaurant} : {e}")
    finally:
        time.sleep(5)
        driver.quit()


# Traitement de tous les restaurants
for creds in CREDENTIALS:
    collect_from_dishop(creds["restaurant"], creds["email"], creds["password"], creds["shop_id"])

if all_data:
    df_new = pd.concat(all_data, ignore_index=True)
    df_new.to_csv(FICHIER_CSV_TEMP, index=False, encoding="utf-8")
    print(f"Données récentes sauvegardées dans : {FICHIER_CSV_TEMP}")
else:
    print("Aucune donnée collectée.")

In [ ]:
# FUSION DISHOP : fusionner la base existante et les nouvelles données

import pandas as pd

FICHIER_CSV_BASE = "BDD_Client_Dishop.csv"
FICHIER_CSV_TEMP = "Dishop_Temp.csv"

# Charger les fichiers
df_old = pd.read_csv(FICHIER_CSV_BASE)
df_new = pd.read_csv(FICHIER_CSV_TEMP)

# Normaliser les colonnes
for col in df_old.columns:
    if col not in df_new.columns:
        df_new[col] = None

df_new = df_new[df_old.columns]

# Fusion
df_combined = pd.concat([df_old, df_new], ignore_index=True)

# Dédoublonnage sur téléphone
colonnes_infos = [col for col in ["Numéro", "Email", "Restaurant", "Prénom"] if col in df_combined.columns]
df_combined["infos"] = df_combined[colonnes_infos].notnull().sum(axis=1)

df_combined = df_combined.sort_values(by="infos", ascending=False)
df_combined = df_combined.drop_duplicates(subset="Numéro", keep="first")
df_combined = df_combined.drop(columns="infos")

# Sauvegarde
df_combined.to_csv(FICHIER_CSV_BASE, index=False, encoding="utf-8")
print("Fusion terminée. Base Dishop mise à jour.")